# GNN-LLM Revamp: GAT + Focal Loss + Optuna

Trains the improved `GNN-11F+LLM` variant with three upgrades over the original:

| Upgrade | Old | New |
|---------|-----|-----|
| **GNN layer** | SAGEConv (unweighted mean aggregation) | GATConv (learned attention, configurable heads) |
| **LLM embeddings** | Topology only — used to build edges, then discarded | Features + weighted edges — 768-dim concatenated to product features; cosine similarity as `edge_attr` on capability edges |
| **Loss** | BCEWithLogitsLoss + pos_weight | Binary Focal Loss (α, γ configurable) |
| **Hyperparameters** | Fixed defaults | Optuna study, 40 trials, train/val only |

**Checkpoint saved to:** `data/models/gnn/checkpoints/gnn_llm_v2.pt`

**Strict leakage rule:** Optuna sees only `train_labels.csv` + `val_labels.csv`. Test set is never touched during tuning.

In [1]:
import os, sys, pickle, warnings, gc
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, to_hetero
from torch.utils.checkpoint import checkpoint as grad_checkpoint
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(0.95)

DATA_DIR  = 'data'
CKPT_DIR  = os.path.join(DATA_DIR, 'models', 'gnn', 'checkpoints')
CKPT_V2   = os.path.join(CKPT_DIR, 'gnn_llm_v2.pt')
VAL_YEAR  = 2013
TEST_YEAR = 2015
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
print(f'Checkpoint will save to: {CKPT_V2}')

Device: cuda
Checkpoint will save to: data\models\gnn\checkpoints\gnn_llm_v2.pt


## Load Artifacts

Builds the augmented product feature tensor: `[5018, 771]` = 3 hand-crafted + 768 LLM embedding dims.  
Also pre-computes cosine similarity weights for all capability edges — stored once, reused every snapshot.

In [2]:
torch.manual_seed(42); np.random.seed(42)

# ── Standard pipeline artifacts ────────────────────────────────────────────────
edge_idx_raw   = torch.load(os.path.join(DATA_DIR, 'edge_index_by_year.pt'), weights_only=False)
edge_idx_by_yr = {k: v.long() for k, v in edge_idx_raw.items()}
p_x_by_yr_base = torch.load(os.path.join(DATA_DIR, 'product_x_by_year.pt'), weights_only=False)
c_x_11feat     = torch.load(os.path.join(DATA_DIR, 'country_x_by_year.pt'), weights_only=False)

with open(os.path.join(DATA_DIR, 'country_mapping.pkl'), 'rb') as f: c_map = pickle.load(f)
with open(os.path.join(DATA_DIR, 'product_mapping.pkl'), 'rb') as f: p_map = pickle.load(f)

train_lbl = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))
val_lbl   = pd.read_csv(os.path.join(DATA_DIR, 'val_labels.csv'))
# test_lbl intentionally not loaded here — never touched during tuning

# ── LLM embeddings → augmented product features ────────────────────────────────
# Concatenate 768-dim LLM embeddings to the 3 hand-crafted product features.
# The LLM embeddings are year-invariant (product descriptions don't change annually),
# so we broadcast the same embedding tensor across all years.
llm_emb = torch.load(
    os.path.join(DATA_DIR, 'product_llm_embeddings.pt'),
    weights_only=False, map_location='cpu'
).float()  # [5018, 768], unit-normalised
print(f'LLM embeddings: {llm_emb.shape}')

p_x_by_yr = {}
for yr, base_feat in p_x_by_yr_base.items():
    # base_feat: [5018, 3] — concat LLM to get [5018, 771]
    p_x_by_yr[yr] = torch.cat([base_feat, llm_emb], dim=1)

P_IN = p_x_by_yr[TEST_YEAR].shape[1]   # 771
C_IN = c_x_11feat[TEST_YEAR].shape[1]  # 11
print(f'Product feature dim (3 + 768): {P_IN}')
print(f'Country feature dim: {C_IN}')

# ── Capability edges + cosine weights ─────────────────────────────────────────
cap_ei = torch.load(
    os.path.join(DATA_DIR, 'capability_edge_index.pt'), weights_only=False
).long()  # [2, 144192]

# Cosine weights: sim(src, dst) for every capability edge.
# Embeddings are unit-normalised so dot product = cosine similarity.
emb_np = llm_emb.numpy()
src_np = cap_ei[0].numpy()
dst_np = cap_ei[1].numpy()
cos_weights = torch.tensor(
    (emb_np[src_np] * emb_np[dst_np]).sum(axis=1),
    dtype=torch.float32
).unsqueeze(1)  # [144192, 1] — shape expected by GATConv edge_dim=1

print(f'Capability edges: {cap_ei.shape}  |  weight range: [{cos_weights.min():.3f}, {cos_weights.max():.3f}]')
print(f'\nTrain {len(train_lbl):,}  Val {len(val_lbl):,}')

LLM embeddings: torch.Size([5018, 768])
Product feature dim (3 + 768): 771
Country feature dim: 11
Capability edges: torch.Size([2, 144192])  |  weight range: [0.319, 1.000]

Train 1,699,206  Val 128,278


## Architecture

### Key changes from original

**`_GATBlock`** replaces `_HomoGNN`:
- Two GATConv layers with `heads` attention heads
- First layer: `concat=True` → output dim = `hidden * heads`, then projected back to `hidden`
- Second layer: `concat=False` (average over heads) → output dim = `hidden`
- `edge_dim=1` so capability edges pass cosine similarity as attention input; trade edges pass `None`

**`BipartiteEncoderGAT`** replaces `BipartiteEncoder`:
- Product projection now `Linear(771, hidden)` (was `Linear(3, hidden)`)
- Forward signature includes `edge_attr_dict` (optional, `None` for non-capability edges)

**`TemporalGNN`** and **`LinkPredictor`** are unchanged.

**`BinaryFocalLoss`**: new loss module replacing `BCEWithLogitsLoss`.

In [3]:
# ── Focal Loss ─────────────────────────────────────────────────────────────────
class BinaryFocalLoss(nn.Module):
    """
    Binary focal loss for imbalanced classification.
    Takes raw logits; applies sigmoid internally.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    alpha : scalar weight for the positive class (balances class frequency)
    gamma : focusing parameter; 0 = standard BCE, higher = more focus on hard examples
    """
    def __init__(self, alpha: float = 0.5, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Numerically stable sigmoid cross-entropy
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = torch.exp(-bce)                       # equivalent to sigmoid(y*logit)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()


# ── GAT block (replaces _HomoGNN) ──────────────────────────────────────────────
class _GATBlock(nn.Module):
    """
    Two-layer GAT.
    Layer 1: concat=True  → dim in * heads, then linear back to hidden
    Layer 2: concat=False → dim hidden (averaged over heads)
    edge_dim=1 accepts the single cosine-similarity scalar per capability edge.
    Trade edges pass edge_attr=None; GAT handles that via fill_value='mean'.
    """
    def __init__(self, hidden: int, heads: int, drop: float):
        super().__init__()
        self.hidden = hidden
        self.heads  = heads
        self.drop   = drop
        self.gat1   = GATConv(hidden, hidden, heads=heads, concat=True,
                               dropout=drop, edge_dim=1,
                               add_self_loops=False, fill_value='mean')
        self.proj   = nn.Linear(hidden * heads, hidden)   # collapse multi-head output
        self.gat2   = GATConv(hidden, hidden, heads=1,  concat=False,
                               dropout=drop, edge_dim=1,
                               add_self_loops=False, fill_value='mean')

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,
                edge_attr: torch.Tensor | None = None) -> torch.Tensor:
        x = F.dropout(self.proj(self.gat1(x, edge_index, edge_attr=edge_attr).relu()),
                      p=self.drop, training=self.training)
        return self.gat2(x, edge_index, edge_attr=edge_attr)


# ── Bipartite encoder ──────────────────────────────────────────────────────────
class BipartiteEncoderGAT(nn.Module):
    def __init__(self, c_in: int, p_in: int, hidden: int, heads: int, drop: float, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_GATBlock(hidden, heads, drop), meta)

    def forward(self, x_dict, ei_dict, ea_dict=None):
        x_proj = {
            'country': self.country_lin(x_dict['country']),
            'product': self.product_lin(x_dict['product']),
        }
        if ea_dict is not None:
            return self.gnn(x_proj, ei_dict, ea_dict)
        return self.gnn(x_proj, ei_dict)


# ── Temporal GNN (unchanged structure, updated encoder type) ───────────────────
class TemporalGNNv2(nn.Module):
    def __init__(self, enc: BipartiteEncoderGAT, hidden: int):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden, batch_first=False)
        self.gru_p = nn.GRU(hidden, hidden, batch_first=False)

    def forward(self, snaps, use_checkpoint: bool = False):
        cs, ps = [], []
        for s in snaps:
            ea = getattr(s, '_ea_dict', None)   # edge_attr_dict stored on snapshot
            if use_checkpoint and self.training:
                def _fwd(xd, eid, ead=ea):
                    return self.enc(xd, eid, ead)
                z = grad_checkpoint(_fwd, s.x_dict, s.edge_index_dict, use_reentrant=False)
            else:
                z = self.enc(s.x_dict, s.edge_index_dict, ea)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}


# ── Link predictor (unchanged) ─────────────────────────────────────────────────
class LinkPredictor(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))

    def forward(self, zc, zp, ei):
        return self.mlp(torch.cat([zc[ei[0]], zp[ei[1]]], dim=-1)).view(-1)


print('Architecture defined.')
print(f'  Product input dim:  {P_IN}  (3 BACI + 768 LLM)')
print(f'  Country input dim:  {C_IN}  (11 BACI+WDI)')

Architecture defined.
  Product input dim:  771  (3 BACI + 768 LLM)
  Country input dim:  11  (11 BACI+WDI)


## Snapshot & Sample Builders

Each snapshot now carries:
- `data['product'].x` — `[5018, 771]` (3 BACI + 768 LLM features)
- `data['product','capability','product'].edge_attr` — `[E, 1]` cosine weights
- `data._ea_dict` — edge_attr dict forwarded to the encoder (trade edges get `None`)

In [4]:
def build_snap_v2(year: int) -> HeteroData:
    d = HeteroData()
    d['country'].x = c_x_11feat[year]          # [233, 11]
    d['product'].x  = p_x_by_yr[year]           # [5018, 771]

    ei = edge_idx_by_yr[year].long()
    d['country', 'exports',     'product'].edge_index = ei
    d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
    d['product', 'capability',  'product'].edge_index = cap_ei
    d['product', 'capability',  'product'].edge_attr  = cos_weights

    # Attach edge_attr_dict so TemporalGNNv2 can pass it to the encoder.
    # Trade edges don't carry weights; None signals GAT to use fill_value='mean'.
    d._ea_dict = {
        ('country', 'exports',     'product'): None,
        ('product', 'rev_exports', 'country'): None,
        ('product', 'capability',  'product'): cos_weights,
    }
    return d


def build_sample_v2(obs_yr: int, ldf: pd.DataFrame) -> dict:
    snaps = [build_snap_v2(y) for y in range(obs_yr - 4, obs_yr + 1)]
    row   = ldf[ldf['year'] == obs_yr].copy().reset_index(drop=True)
    ci_s  = row['country'].map(c_map['to_idx'])
    pi_s  = row['product'].map(p_map['to_idx'])
    ok    = ci_s.notna() & pi_s.notna()
    ci    = ci_s[ok].astype(int).values
    pi    = pi_s[ok].astype(int).values
    return {
        'snapshots': snaps,
        'labels': {
            'edge_label_index': torch.tensor([ci, pi], dtype=torch.long),
            'edge_label':       torch.tensor(row.loc[ok, 'label'].values, dtype=torch.float32),
        },
        'year':          int(obs_yr),
        'countries_raw': row.loc[ok, 'country'].values,
        'products_raw':  row.loc[ok, 'product'].values,
    }


def to_dev(samp: dict, dev: str):
    for s in samp['snapshots']:
        s['country'].x = s['country'].x.to(dev)
        s['product'].x  = s['product'].x.to(dev)
        for et in s.edge_types:
            s[et].edge_index = s[et].edge_index.to(device=dev, dtype=torch.long)
            if s[et].get('edge_attr') is not None:
                s[et].edge_attr = s[et].edge_attr.to(dev)
        if s._ea_dict is not None:
            s._ea_dict = {
                k: (v.to(dev) if v is not None else None)
                for k, v in s._ea_dict.items()
            }
    samp['labels']['edge_label_index'] = samp['labels']['edge_label_index'].to(dev)
    samp['labels']['edge_label']       = samp['labels']['edge_label'].to(dev)


def from_dev(samp: dict):
    for s in samp['snapshots']:
        s['country'].x = s['country'].x.cpu()
        s['product'].x  = s['product'].x.cpu()
        for et in s.edge_types:
            s[et].edge_index = s[et].edge_index.cpu()
            if s[et].get('edge_attr') is not None:
                s[et].edge_attr = s[et].edge_attr.cpu()
        if s._ea_dict is not None:
            s._ea_dict = {
                k: (v.cpu() if v is not None else None)
                for k, v in s._ea_dict.items()
            }
    samp['labels']['edge_label_index'] = samp['labels']['edge_label_index'].cpu()
    samp['labels']['edge_label']       = samp['labels']['edge_label'].cpu()


@torch.no_grad()
def get_val_prauc(mdl: TemporalGNNv2, pred: LinkPredictor, va: dict, dev: str) -> float:
    mdl.eval(); pred.eval()
    to_dev(va, dev)
    z      = mdl(va['snapshots'], use_checkpoint=False)
    scores = torch.sigmoid(
        pred(z['country'], z['product'], va['labels']['edge_label_index'])
    ).cpu().numpy()
    labels = va['labels']['edge_label'].cpu().numpy()
    from_dev(va)
    p, r, _ = precision_recall_curve(labels, scores)
    return float(auc(r, p))


print('Snapshot builders ready.')

# Quick smoke-test: build one snapshot and verify shapes
s0 = build_snap_v2(2010)
print(f'  country.x:  {s0["country"].x.shape}   (expected [233, 11])')
print(f'  product.x:  {s0["product"].x.shape}  (expected [5018, 771])')
print(f'  cap edge_attr: {s0["product","capability","product"].edge_attr.shape}  (expected [144192, 1])')

Snapshot builders ready.
  country.x:  torch.Size([233, 11])   (expected [233, 11])
  product.x:  torch.Size([5018, 771])  (expected [5018, 771])
  cap edge_attr: torch.Size([144192, 1])  (expected [144192, 1])


## Pre-Build Datasets

Build train and val sample lists once — reused across all Optuna trials to avoid redundant IO.

In [5]:
print('Building training samples (one per observation year)...')
tr_samples = [
    build_sample_v2(yr, train_lbl)
    for yr in sorted(train_lbl['year'].unique())
]
print(f'  Training samples: {len(tr_samples)} years')

print(f'Building validation sample (year {VAL_YEAR})...')
va_sample = build_sample_v2(VAL_YEAR, val_lbl)
print(f'  Val pairs: {len(va_sample["labels"]["edge_label"]):,}')
print(f'  Val positive rate: {va_sample["labels"]["edge_label"].mean()*100:.1f}%')

Building training samples (one per observation year)...
  Training samples: 13 years
Building validation sample (year 2013)...
  Val pairs: 128,278
  Val positive rate: 14.8%


## Optuna Hyperparameter Search

**Search space:**

| Parameter | Distribution |
|-----------|-------------|
| `lr` | log-uniform [1e-4, 1e-2] |
| `weight_decay` | log-uniform [1e-6, 1e-3] |
| `hidden_dim` | categorical {64, 128} |
| `dropout` | uniform [0.1, 0.5] |
| `gat_heads` | categorical {2, 4} |
| `focal_alpha` | uniform [0.25, 0.85] |
| `focal_gamma` | uniform [1.0, 3.0] |

**Objective:** maximise validation PR-AUC.  
**Constraint:** train on `train_labels` only, evaluate on `val_labels` only. Test set untouched.

In [6]:
N_TRIALS     = 40
TRIAL_EPOCHS = 30   # shortened epochs per trial; full training uses best params
TRIAL_PATIENCE = 8

def build_model(hidden: int, heads: int, drop: float, meta) -> tuple:
    enc  = BipartiteEncoderGAT(C_IN, P_IN, hidden, heads, drop, meta)
    mdl  = TemporalGNNv2(enc, hidden)
    pred = LinkPredictor(hidden)
    return mdl, pred


def run_trial_training(
    mdl, pred, crit, opt,
    tr: list, va: dict,
    epochs: int, patience: int,
    dev: str,
    trial: optuna.Trial | None = None,
) -> float:
    """Train for up to `epochs` with early stopping. Returns best val PR-AUC."""
    best_vpa, no_imp = -1.0, 0
    best_state = None

    for ep in range(1, epochs + 1):
        mdl.train(); pred.train()
        ep_loss = 0.0
        for samp in tr:
            to_dev(samp, dev)
            opt.zero_grad()
            z    = mdl(samp['snapshots'], use_checkpoint=True)
            loss = crit(
                pred(z['country'], z['product'], samp['labels']['edge_label_index']),
                samp['labels']['edge_label']
            )
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(mdl.parameters()) + list(pred.parameters()), 1.0
            )
            opt.step()
            ep_loss += loss.item()
            from_dev(samp)
            del z, loss
            torch.cuda.empty_cache()

        vpa = get_val_prauc(mdl, pred, va, dev)

        if vpa > best_vpa:
            best_vpa   = vpa
            best_state = (
                {k: v.cpu().clone() for k, v in mdl.state_dict().items()},
                {k: v.cpu().clone() for k, v in pred.state_dict().items()},
            )
            no_imp = 0
        else:
            no_imp += 1

        # Optuna pruning — report intermediate value
        if trial is not None:
            trial.report(vpa, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()

        if no_imp >= patience:
            break

    return best_vpa, best_state


def objective(trial: optuna.Trial) -> float:
    # Sample hyperparameters
    hidden   = trial.suggest_categorical('hidden_dim',    [64, 128])
    heads    = trial.suggest_categorical('gat_heads',     [2, 4])
    drop     = trial.suggest_float('dropout',             0.1, 0.5)
    lr       = trial.suggest_float('lr',                  1e-4, 1e-2, log=True)
    wd       = trial.suggest_float('weight_decay',        1e-6, 1e-3, log=True)
    f_alpha  = trial.suggest_float('focal_alpha',         0.25, 0.85)
    f_gamma  = trial.suggest_float('focal_gamma',         1.0,  3.0)

    torch.manual_seed(42)
    meta = tr_samples[0]['snapshots'][0].metadata()
    mdl, pred = build_model(hidden, heads, drop, meta)
    mdl  = mdl.to(DEVICE)
    pred = pred.to(DEVICE)

    crit = BinaryFocalLoss(alpha=f_alpha, gamma=f_gamma)
    opt  = torch.optim.Adam(
        list(mdl.parameters()) + list(pred.parameters()),
        lr=lr, weight_decay=wd
    )

    best_vpa, _ = run_trial_training(
        mdl, pred, crit, opt,
        tr_samples, va_sample,
        epochs=TRIAL_EPOCHS, patience=TRIAL_PATIENCE,
        dev=DEVICE, trial=trial,
    )

    # Clean up GPU memory before next trial
    del mdl, pred, crit, opt
    torch.cuda.empty_cache(); gc.collect()

    return best_vpa


print(f'Starting Optuna study: {N_TRIALS} trials × {TRIAL_EPOCHS} epochs each...')
print(f'Sampler: TPE  |  Pruner: MedianPruner')
print(f'Objective: maximise validation PR-AUC (train/val only — test untouched)')

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f'\nBest trial #{best.number}  val PR-AUC = {best.value:.4f}')
print('Best hyperparameters:')
for k, v in best.params.items():
    print(f'  {k}: {v}')

Starting Optuna study: 40 trials × 30 epochs each...
Sampler: TPE  |  Pruner: MedianPruner
Objective: maximise validation PR-AUC (train/val only — test untouched)


  0%|          | 0/40 [00:00<?, ?it/s]


Best trial #15  val PR-AUC = 0.3963
Best hyperparameters:
  hidden_dim: 64
  gat_heads: 2
  dropout: 0.28102624807606014
  lr: 0.005108000941674704
  weight_decay: 2.855864788722317e-06
  focal_alpha: 0.5429020471094602
  focal_gamma: 1.6869548937276326


## Final Training with Best Hyperparameters

Re-trains from scratch using the best parameters found by Optuna, now with full epochs (80) and patience (15).  
Checkpoint saved as `gnn_llm_v2.pt`.

In [7]:
torch.manual_seed(42); np.random.seed(42)

bp = best.params
FINAL_EPOCHS   = 80
FINAL_PATIENCE = 15

print('Final training with best hyperparameters:')
print(f'  hidden={bp["hidden_dim"]}  heads={bp["gat_heads"]}  dropout={bp["dropout"]:.3f}')
print(f'  lr={bp["lr"]:.2e}  wd={bp["weight_decay"]:.2e}')
print(f'  focal_alpha={bp["focal_alpha"]:.3f}  focal_gamma={bp["focal_gamma"]:.3f}')

meta = tr_samples[0]['snapshots'][0].metadata()
mdl_final, pred_final = build_model(
    bp['hidden_dim'], bp['gat_heads'], bp['dropout'], meta
)
mdl_final  = mdl_final.to(DEVICE)
pred_final = pred_final.to(DEVICE)

crit_final = BinaryFocalLoss(alpha=bp['focal_alpha'], gamma=bp['focal_gamma'])
opt_final  = torch.optim.Adam(
    list(mdl_final.parameters()) + list(pred_final.parameters()),
    lr=bp['lr'], weight_decay=bp['weight_decay']
)

print(f'\nTraining for up to {FINAL_EPOCHS} epochs (patience={FINAL_PATIENCE})...')

best_vpa_final = -1.0
best_state_final = None
no_imp = 0

for ep in range(1, FINAL_EPOCHS + 1):
    mdl_final.train(); pred_final.train()
    ep_loss = 0.0
    for samp in tr_samples:
        to_dev(samp, DEVICE)
        opt_final.zero_grad()
        z    = mdl_final(samp['snapshots'], use_checkpoint=True)
        loss = crit_final(
            pred_final(z['country'], z['product'], samp['labels']['edge_label_index']),
            samp['labels']['edge_label']
        )
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(mdl_final.parameters()) + list(pred_final.parameters()), 1.0
        )
        opt_final.step()
        ep_loss += loss.item()
        from_dev(samp)
        del z, loss
        torch.cuda.empty_cache()

    vpa = get_val_prauc(mdl_final, pred_final, va_sample, DEVICE)

    if vpa > best_vpa_final:
        best_vpa_final = vpa
        best_state_final = (
            {k: v.cpu().clone() for k, v in mdl_final.state_dict().items()},
            {k: v.cpu().clone() for k, v in pred_final.state_dict().items()},
        )
        no_imp = 0
    else:
        no_imp += 1

    if ep % 10 == 0 or no_imp == 0:
        print(f'  Ep {ep:3d}  loss={ep_loss/len(tr_samples):.4f}  '
              f'val_PR-AUC={vpa:.4f}  best={best_vpa_final:.4f}')

    if no_imp >= FINAL_PATIENCE:
        print(f'  Early stop at epoch {ep}  (best val PR-AUC={best_vpa_final:.4f})')
        break

# Restore best weights
mdl_final.load_state_dict({k: v.to(DEVICE) for k, v in best_state_final[0].items()})
pred_final.load_state_dict({k: v.to(DEVICE) for k, v in best_state_final[1].items()})

print(f'\nFinal best val PR-AUC: {best_vpa_final:.4f}')

Final training with best hyperparameters:
  hidden=64  heads=2  dropout=0.281
  lr=5.11e-03  wd=2.86e-06
  focal_alpha=0.543  focal_gamma=1.687

Training for up to 80 epochs (patience=15)...
  Ep   1  loss=0.0788  val_PR-AUC=0.2542  best=0.2542
  Ep   2  loss=0.0661  val_PR-AUC=0.2820  best=0.2820
  Ep   3  loss=0.0640  val_PR-AUC=0.3005  best=0.3005
  Ep   4  loss=0.0628  val_PR-AUC=0.3038  best=0.3038
  Ep   5  loss=0.0630  val_PR-AUC=0.3111  best=0.3111
  Ep   6  loss=0.0618  val_PR-AUC=0.3189  best=0.3189
  Ep   7  loss=0.0619  val_PR-AUC=0.3193  best=0.3193
  Ep   8  loss=0.0618  val_PR-AUC=0.3267  best=0.3267
  Ep   9  loss=0.0611  val_PR-AUC=0.3333  best=0.3333
  Ep  10  loss=0.0609  val_PR-AUC=0.3393  best=0.3393
  Ep  11  loss=0.0609  val_PR-AUC=0.3411  best=0.3411
  Ep  12  loss=0.0608  val_PR-AUC=0.3466  best=0.3466
  Ep  14  loss=0.0604  val_PR-AUC=0.3591  best=0.3591
  Ep  15  loss=0.0600  val_PR-AUC=0.3685  best=0.3685
  Ep  18  loss=0.0598  val_PR-AUC=0.3699  best=0.3699

## Save Checkpoint

Saves everything needed to reconstruct and run the model at inference time, including the
best hyperparameters and the product feature dimensionality so loaders don't need to guess.

In [8]:
meta = tr_samples[0]['snapshots'][0].metadata()

torch.save({
    # Model weights
    'mdl_state':   {k: v.cpu() for k, v in mdl_final.state_dict().items()},
    'pred_state':  {k: v.cpu() for k, v in pred_final.state_dict().items()},
    # Architecture config needed to reconstruct
    'c_in':        C_IN,
    'p_in':        P_IN,
    'hidden':      bp['hidden_dim'],
    'heads':       bp['gat_heads'],
    'dropout':     bp['dropout'],
    'meta':        meta,
    # Training config — for reference
    'focal_alpha': bp['focal_alpha'],
    'focal_gamma': bp['focal_gamma'],
    'best_val_prauc': best_vpa_final,
    'best_hparams':   dict(bp),
}, CKPT_V2)

print(f'Checkpoint saved -> {CKPT_V2}')
print(f'File size: {os.path.getsize(CKPT_V2)/1e6:.1f} MB')

Checkpoint saved -> data\models\gnn\checkpoints\gnn_llm_v2.pt
File size: 0.7 MB


## Unoptimized Baseline: Same Architecture, Fixed Hyperparameters

Trains the identical GAT + LLM architecture (`gnn_llm_v2_unopt.pt`) using fixed defaults — no Optuna tuning.  
Everything else is unchanged: LLM product features, cosine capability edge weights, focal loss, same train/val split.

In [11]:
CKPT_UNOPT = os.path.join(CKPT_DIR, 'gnn_llm_v2_unopt.pt')

# Fixed default hyperparameters — no tuning
UNOPT_PARAMS = {
    'hidden_dim':   128,
    'gat_heads':    4,
    'dropout':      0.3,
    'lr':           1e-3,
    'weight_decay': 1e-4,
    'focal_alpha':  0.5,
    'focal_gamma':  2.0,
}
UNOPT_EPOCHS   = 80
UNOPT_PATIENCE = 30

print('Unoptimized training with fixed hyperparameters:')
for k, v in UNOPT_PARAMS.items():
    print(f'  {k}: {v}')

torch.manual_seed(42); np.random.seed(42)

meta_u = tr_samples[0]['snapshots'][0].metadata()
mdl_unopt, pred_unopt = build_model(
    UNOPT_PARAMS['hidden_dim'], UNOPT_PARAMS['gat_heads'], UNOPT_PARAMS['dropout'], meta_u
)
mdl_unopt  = mdl_unopt.to(DEVICE)
pred_unopt = pred_unopt.to(DEVICE)

crit_unopt = BinaryFocalLoss(alpha=UNOPT_PARAMS['focal_alpha'], gamma=UNOPT_PARAMS['focal_gamma'])
opt_unopt  = torch.optim.Adam(
    list(mdl_unopt.parameters()) + list(pred_unopt.parameters()),
    lr=UNOPT_PARAMS['lr'], weight_decay=UNOPT_PARAMS['weight_decay']
)

print(f'\nTraining for up to {UNOPT_EPOCHS} epochs (patience={UNOPT_PATIENCE})...')

best_vpa_unopt   = -1.0
best_state_unopt = None
no_imp_u         = 0

for ep in range(1, UNOPT_EPOCHS + 1):
    mdl_unopt.train(); pred_unopt.train()
    ep_loss = 0.0
    for samp in tr_samples:
        to_dev(samp, DEVICE)
        opt_unopt.zero_grad()
        z    = mdl_unopt(samp['snapshots'], use_checkpoint=True)
        loss = crit_unopt(
            pred_unopt(z['country'], z['product'], samp['labels']['edge_label_index']),
            samp['labels']['edge_label']
        )
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(mdl_unopt.parameters()) + list(pred_unopt.parameters()), 1.0
        )
        opt_unopt.step()
        ep_loss += loss.item()
        from_dev(samp)
        del z, loss
        torch.cuda.empty_cache()

    vpa = get_val_prauc(mdl_unopt, pred_unopt, va_sample, DEVICE)

    if vpa > best_vpa_unopt:
        best_vpa_unopt = vpa
        best_state_unopt = (
            {k: v.cpu().clone() for k, v in mdl_unopt.state_dict().items()},
            {k: v.cpu().clone() for k, v in pred_unopt.state_dict().items()},
        )
        no_imp_u = 0
    else:
        no_imp_u += 1

    if ep % 10 == 0 or no_imp_u == 0:
        print(f'  Ep {ep:3d}  loss={ep_loss/len(tr_samples):.4f}  '
              f'val_PR-AUC={vpa:.4f}  best={best_vpa_unopt:.4f}')

    if no_imp_u >= UNOPT_PATIENCE:
        print(f'  Early stop at epoch {ep}  (best val PR-AUC={best_vpa_unopt:.4f})')
        break

# Restore best weights
mdl_unopt.load_state_dict({k: v.to(DEVICE) for k, v in best_state_unopt[0].items()})
pred_unopt.load_state_dict({k: v.to(DEVICE) for k, v in best_state_unopt[1].items()})

print(f'\nFinal best val PR-AUC (unoptimized): {best_vpa_unopt:.4f}')

# Save checkpoint
torch.save({
    'mdl_state':      {k: v.cpu() for k, v in mdl_unopt.state_dict().items()},
    'pred_state':     {k: v.cpu() for k, v in pred_unopt.state_dict().items()},
    'c_in':           C_IN,
    'p_in':           P_IN,
    'hidden':         UNOPT_PARAMS['hidden_dim'],
    'heads':          UNOPT_PARAMS['gat_heads'],
    'dropout':        UNOPT_PARAMS['dropout'],
    'meta':           meta_u,
    'focal_alpha':    UNOPT_PARAMS['focal_alpha'],
    'focal_gamma':    UNOPT_PARAMS['focal_gamma'],
    'best_val_prauc': best_vpa_unopt,
    'hparams':        UNOPT_PARAMS,
    'optimized':      False,
}, CKPT_UNOPT)

print(f'Checkpoint saved -> {CKPT_UNOPT}')
print(f'File size: {os.path.getsize(CKPT_UNOPT)/1e6:.1f} MB')
print(f'\nComparison:')
print(f'  Optimized   (gnn_llm_v2.pt):       val PR-AUC = {best_vpa_final:.4f}')
print(f'  Unoptimized (gnn_llm_v2_unopt.pt): val PR-AUC = {best_vpa_unopt:.4f}')

Unoptimized training with fixed hyperparameters:
  hidden_dim: 128
  gat_heads: 4
  dropout: 0.3
  lr: 0.001
  weight_decay: 0.0001
  focal_alpha: 0.5
  focal_gamma: 2.0

Training for up to 80 epochs (patience=30)...
  Ep   1  loss=0.0652  val_PR-AUC=0.2122  best=0.2122
  Ep   2  loss=0.0560  val_PR-AUC=0.2620  best=0.2620
  Ep   3  loss=0.0532  val_PR-AUC=0.2685  best=0.2685
  Ep   4  loss=0.0523  val_PR-AUC=0.2734  best=0.2734
  Ep   5  loss=0.0515  val_PR-AUC=0.2895  best=0.2895
  Ep   6  loss=0.0510  val_PR-AUC=0.2961  best=0.2961
  Ep   8  loss=0.0505  val_PR-AUC=0.3009  best=0.3009
  Ep  10  loss=0.0503  val_PR-AUC=0.3059  best=0.3059
  Ep  11  loss=0.0504  val_PR-AUC=0.3079  best=0.3079
  Ep  16  loss=0.0501  val_PR-AUC=0.3124  best=0.3124
  Ep  19  loss=0.0499  val_PR-AUC=0.3156  best=0.3156
  Ep  20  loss=0.0500  val_PR-AUC=0.3137  best=0.3156
  Ep  23  loss=0.0500  val_PR-AUC=0.3159  best=0.3159
  Ep  24  loss=0.0505  val_PR-AUC=0.3160  best=0.3160
  Ep  29  loss=0.0500  val_

## Quick Validation-Set Summary

Sanity-check metrics on val set only. Full test-set evaluation belongs in `evaluation.ipynb` / `full_universe_eval.ipynb`.

In [13]:
mdl_unopt.eval(); pred_final.eval()
to_dev(va_sample, DEVICE)

with torch.no_grad():
    z_val    = mdl_unopt(va_sample['snapshots'], use_checkpoint=False)
    scores_v = torch.sigmoid(
        pred_final(z_val['country'], z_val['product'],
                   va_sample['labels']['edge_label_index'])
    ).cpu().numpy()

labels_v = va_sample['labels']['edge_label'].cpu().numpy()
from_dev(va_sample)

p_v, r_v, _ = precision_recall_curve(labels_v, scores_v)
prauc_v = auc(r_v, p_v)
auroc_v = roc_auc_score(labels_v, scores_v)

print('Validation-set metrics (GNN-LLM v2):')
print(f'  PR-AUC : {prauc_v:.4f}')
print(f'  AUROC  : {auroc_v:.4f}')
print()
print('Best hyperparameters found by Optuna:')
for k, v in best.params.items():
    print(f'  {k:20s}: {v}')
print(f'\nOptuna study: {len(study.trials)} trials completed')
pruned  = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
print(f'  Completed: {complete}  Pruned: {pruned}')

# Hyperparameter importance (requires optuna[visualization])
try:
    importances = optuna.importance.get_param_importances(study)
    print('\nHyperparameter importances:')
    for param, imp in importances.items():
        print(f'  {param:20s}: {imp:.3f}')
except Exception:
    pass

print('\nTo run full evaluation on the test set, load this checkpoint in evaluation.ipynb')
print(f'  ckpt = torch.load("{CKPT_V2}", weights_only=False)')
print(f'  # then use BipartiteEncoderGAT + TemporalGNNv2 + p_in=771, c_in=11')

RuntimeError: mat1 and mat2 shapes cannot be multiplied (128278x256 and 128x64)